# `lake` demo

Thin demo that imports the package and reproduces the original SHAP / scatter plots.
Every figure produced here is also written to `outputs/figures/`.


In [ ]:
# Standard imports + the lake package.
import pandas as pd
from lake import default_config
from lake.data.dataset import load_dataset, normalize, split_timeseries_by_lake
from lake.models.tree import train_tree_regressor
from lake.explain.shap_utils import shap_explain_tree, save_shap_table
from lake.viz.shap_plots import save_shap_summary_plot
from lake.viz.scatter import save_pred_vs_true_scatter
from lake.config import FEATURE_COLUMNS

cfg = default_config(exp_name="demo")
cfg.paths.ensure()
cfg.training.debug_row_limit = 1000   # keep the demo fast

In [ ]:
# 1. Load + split + normalize.
df = load_dataset(cfg.paths.data_csv, debug_row_limit=cfg.training.debug_row_limit)
splits = split_timeseries_by_lake(df)
splits_s, scaler = normalize(splits)
print({k: getattr(splits_s, k).X.shape for k in ('train', 'valid', 'test')})

In [ ]:
# 2. Train an XGBoost regressor and plot measured-vs-predicted.
ytr_pred, yte_pred, fitted = train_tree_regressor(
    splits_s.train.X, splits_s.train.y,
    splits_s.test.X, splits_s.test.y,
    kind='XGB',
)

save_pred_vs_true_scatter(
    [('train', splits_s.train.y, ytr_pred), ('test', splits_s.test.y, yte_pred)],
    fig_path=cfg.paths.figures_dir / 'scatter_demo.png',
)

In [ ]:
# 3. Compute averaged SHAP values on the test split and save them.
shap_values = shap_explain_tree(fitted, splits_s.test.X, num_iter=10)
save_shap_table(
    shap_values,
    feature_names=FEATURE_COLUMNS,
    output_csv=cfg.paths.shap_dir / 'demo.csv',
    extra_columns={'ID': splits_s.test.ids[:, 0], 'year': splits_s.test.ids[:, 1]},
)
save_shap_summary_plot(
    shap_values, splits_s.test.X, FEATURE_COLUMNS,
    fig_path=cfg.paths.figures_dir / 'shap_summary_demo.png',
)